# Score and cost prediction: kNN

Compare **TF-IDF** and **E5 + four features (388 dimensions)** using training
data only. For each prompt, predict a five-class score distribution and a
dollar cost for each of the 11 candidate LLMs.

**Workflow:** load train → three-fold grouped CV → fit selected parameters
on train → report training errors → save models and predictions.

No calibration, validation, or test files are read. CV chooses parameters
within each representation; final training errors describe the fitted
models and do not establish performance on new prompts or select a model family.

Run top to bottom with `RUN_MODE = "smoke"` first, then use `"full"`.
Smoke uses about 512 training prompts with the same CV grids; its results
are execution checks. Full uses all prepared training prompts.

The final fitted models are saved for a separate inference notebook. Only training-row predictions and training-fold CV predictions are computed here.

**Execution layout:** CV and final training each have separate **TF-IDF** and **E5_388** code cells, with visible headings. Shared helper cells only define computations. After setup, each execution cell processes its named representation only. Rerunning CV replaces that representation's CV tables and clears its old fitted model; rerun its final training, then the shared diagnostics and export. The other representation's results are retained.

## Call this model from the next notebook

After completing a **full** run, copy the `predict_saved_knn` helper defined at the end of this notebook into the next notebook. Then load the saved package as below; no fitting or CV is required. The example path assumes the next notebook's working directory is the repository root; adjust `run_dir` if needed.

**Example for the next notebook (not executed here):**

```python
from pathlib import Path
import joblib
import numpy as np

# Set this to this method's completed full-run directory.
run_dir = Path("2_Router_Training/experiment_outputs/router_training_score_and_cost_kNN/full")
artifact = joblib.load(run_dir / "tfidf_model.joblib")
assert artifact["run_mode"] == "full"
X_new = artifact["preprocessing"]["tfidf_vectorizer"].transform(new_prompt_texts).astype(np.float32)
predictions = predict_saved_knn(artifact, X_new)
llm_names = artifact["model_names"]
```

For E5, load `e5_388_model.joblib` and supply 388-dimensional features
produced by the same Step 1 encoding/chunking recipe. Keep the first 384
semantic coordinates; transform the raw length and two count columns
using the **saved** scalers, then append the code indicator. Do not refit
preprocessing on the new prompts or standardize already scaled columns again.

The helper returns `probability` (rows × 11 × 5), `hard_score`,
`expected_score`, and `dollar_cost` (each rows × 11). Rows follow the input
order and LLM columns follow `artifact["model_names"]`.

## 1. Setup and training data

Required packages: `numpy`, `pandas`, `scipy`, `scikit-learn>=1.6`,
`joblib`, `threadpoolctl`, and `ipykernel`. Use the Step 1 environment when
possible so saved preprocessing objects have compatible versions.

These notebooks are in the project root, beside `data/`. Both input paths
are relative to that root: `data/processed/char_tfidf_50k/` for TF-IDF and
`data/processed/multilingual_e5_small_chunk480/` for E5.

The code automatically finds this root from the kernel's working directory
or its parents. This also works if an existing kernel still starts inside
`2_Router_Training/`. If necessary, set `DATA_PREPARATION_DIR` to the folder
containing `data/`, or set the `ERDOS_DATA_DIR` environment variable.
Outputs remain under `2_Router_Training/experiment_outputs/`; the opening
model-loading examples use that same location.

Inputs are the three `train_*.pkl` tables and `X_train.npz` in
`data/processed/char_tfidf_50k/`, plus `X_train_e5_full_features.npy`
and the two scalers in `data/processed/multilingual_e5_small_chunk480/`.
The TF-IDF preprocessing object supplies feature and LLM column order.
No E5 encoding is rerun.

TF-IDF uses only its text columns. E5 uses 384 semantic coordinates plus
standardized prompt length, choice-marker count, numeric-token count,
and a code indicator. The appended 388-dimensional vector is used as saved.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
import json
import os

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from scipy.sparse import load_npz
from sklearn.base import clone
from sklearn.model_selection import GroupKFold

from sklearn.neighbors import NearestNeighbors

In [2]:
RUN_MODE = "full"  # Use "smoke" for a small end-to-end check.
DATA_PREPARATION_DIR = None  # Auto-find the project root containing data/; override only if needed.
OUTPUT_BASE_OVERRIDE = None  # Optional method-specific output base.
SEED = 42
WORKFLOW = "router_training_score_and_cost_kNN"
score_support = np.array([0.0, 0.25, 0.5, 0.75, 1.0])
NUM_CLASSES = len(score_support)
assert RUN_MODE in {"smoke", "full"}

def find_data_dir():
    requested = DATA_PREPARATION_DIR or os.environ.get("ERDOS_DATA_DIR")
    roots = [Path(requested).expanduser()] if requested else [
        candidate for parent in (Path.cwd(), *Path.cwd().parents)
        for candidate in (parent, parent / "1_data_preparation")
    ]
    required = ("char_tfidf_50k/preprocessing.joblib", "multilingual_e5_small_chunk480/X_train_e5_full_features.npy",
                "multilingual_e5_small_chunk480/length_scaler.joblib", "multilingual_e5_small_chunk480/count_scaler.joblib")
    for root in roots:
        if all((root / "data/processed" / name).is_file() for name in required):
            return root.resolve()
    raise FileNotFoundError("Set DATA_PREPARATION_DIR to the folder containing data/.")

# The notebooks now live beside data/ in the project root.
# Resolve from the kernel working directory or its parents, including an older subfolder kernel.
preparation_dir = find_data_dir()
processed_dir = preparation_dir / "data" / "processed"
table_dir = processed_dir / "char_tfidf_50k"
e5_dir = processed_dir / "multilingual_e5_small_chunk480"
project_dir = preparation_dir.parent if preparation_dir.name == "1_data_preparation" else preparation_dir
output_base = (Path(OUTPUT_BASE_OVERRIDE).expanduser() if OUTPUT_BASE_OVERRIDE else
               project_dir / "2_Router_Training/experiment_outputs" / WORKFLOW)
OUTPUT_DIR = (output_base / RUN_MODE).resolve()
run_started = datetime.now(timezone.utc).isoformat()
print("Mode:", RUN_MODE, "\nPrepared data:", preparation_dir, "\nOutputs:", OUTPUT_DIR)

prediction_bundles, fitted_models, cv_tables, selected_parameters = {}, {}, {}, {}
training_diagnostics_current = False

Mode: full 
Prepared data: C:\Users\johnh\MachineLearning\project_erdos 
Outputs: C:\Users\johnh\MachineLearning\project_erdos\2_Router_Training\experiment_outputs\router_training_score_and_cost_kNN\full


In [3]:
preprocessing = joblib.load(table_dir / "preprocessing.joblib")
tfidf_vectorizer = preprocessing["tfidf_vectorizer"]
model_names = list(preprocessing["model_names"])
length_scaler = joblib.load(e5_dir / "length_scaler.joblib")
count_scaler = joblib.load(e5_dir / "count_scaler.joblib")
train_questions, train_scores, train_costs = [
    pd.read_pickle(table_dir / f"train_{name}.pkl")
    for name in ("questions", "scores", "costs")
]
X_train = {
    "tfidf": load_npz(table_dir / "X_train.npz")[:, :len(tfidf_vectorizer.vocabulary_)].astype(np.float32).tocsr(),
    "e5_388": np.load(e5_dir / "X_train_e5_full_features.npy", allow_pickle=False).astype(np.float32),
}
assert len(model_names) == 11
assert train_questions.index.is_unique and train_questions["sample_id"].is_unique
assert train_questions.index.equals(train_scores.index) and train_scores.index.equals(train_costs.index)
assert train_scores.columns.tolist() == train_costs.columns.tolist() == model_names
raw_scores = train_scores.to_numpy(dtype=float)
assert np.isfinite(raw_scores).all() and ((raw_scores >= 0) & (raw_scores <= 1)).all()
raw_costs = train_costs.to_numpy(dtype=float)
assert np.isfinite(raw_costs).all() and (raw_costs > 0).all()
assert X_train["e5_388"].shape == (len(train_questions), 388)
assert X_train["tfidf"].shape[0] == len(train_questions)
assert np.isfinite(X_train["e5_388"]).all() and np.isfinite(X_train["tfidf"].data).all()

# Coverage describes the original training scores before smoke subsampling.
anchor_counts = [(raw_scores == value).sum() for value in score_support]
coverage = pd.DataFrame({"Recorded score": [*score_support, "Other"],
                        "Count": [*anchor_counts, raw_scores.size - sum(anchor_counts)]})
coverage["Share"] = coverage["Count"] / raw_scores.size
display(coverage)
if RUN_MODE == "smoke":
    positions = np.random.default_rng(SEED).choice(len(train_questions), min(512, len(train_questions)), replace=False)
    examples = [np.argwhere(raw_scores == value)[0, 0] for value in np.unique(raw_scores)]
    positions = np.unique(np.r_[positions, examples])
    train_questions, train_scores, train_costs = [
        frame.iloc[positions].copy() for frame in (train_questions, train_scores, train_costs)
    ]
    X_train = {name: values[positions] for name, values in X_train.items()}
train_score_values = train_scores.to_numpy(dtype=float)
train_cost_values = train_costs.to_numpy(dtype=float)
print("Training prompts:", len(train_questions), "| Feature shapes:", {k: v.shape for k, v in X_train.items()})

# Shared score labels and grouped folds for both representations.
train_class_ids = np.abs(train_score_values[..., None] - score_support).argmin(axis=-1)
groups = train_questions["question_group"].to_numpy()
assert pd.notna(groups).all()
folds = list(GroupKFold(n_splits=3, shuffle=True, random_state=SEED).split(train_questions, groups=groups))
fold_ids = np.zeros(len(train_questions), dtype=int)
for fold_number, (fit_rows, heldout_rows) in enumerate(folds, 1):
    fold_ids[heldout_rows] = fold_number

,Recorded score,Count,Share
0,0.0,93268,0.387165
1,0.25,15064,0.062532
2,0.5,8075,0.033520
3,0.75,27679,0.114898
4,1.0,96473,0.400469
5,Other,341,0.001416


Training prompts: 21900 | Feature shapes: {'tfidf': (21900, 50000), 'e5_388': (21900, 388)}


## 2. Targets, CV, and error definitions

Let $N$ be the number of training prompts, $M=11$ the candidate LLMs,
$x_i$ a prompt's feature vector, and $s_{im},c_{im}$ its recorded score and
dollar cost for LLM $m$. The five score anchors are
$(a_0,\ldots,a_4)=(0,0.25,0.5,0.75,1)$.

Map a recorded score to its nearest anchor:
$z_{im}=\arg\min_{j\in\{0,\ldots,4\}}|s_{im}-a_j|$.
Exact midpoint ties choose the lower anchor. The coverage table explains
the five-class choice; rare intermediate values are approximated for
classification. Numerical errors always use the **original scores**.

Given probabilities $p_{imj}$, the two score outputs are
$$\widehat z_{im}=\arg\max_j p_{imj},\qquad
\widehat s^{\rm hard}_{im}=a_{\widehat z_{im}},\qquad
\widehat s^{\rm expected}_{im}=\sum_{j=0}^4 a_jp_{imj}.$$
Hard-score probability ties choose the lower class. Both outputs share
one probability model and therefore the same selected hyperparameter.

Use three shuffled `GroupKFold` folds (seed 42), grouped by
`question_group`. Each training prompt receives one out-of-fold (OOF)
prediction. TF-IDF and the E5 auxiliary scalers are refitted on the
fitting part of each fold; E5's 384 semantic coordinates remain frozen.
Final fits use the prepared full-training features.

All reported summaries give equal weight to the 11 LLMs. For any evaluated
set of $n$ prompts, with $\mathbf1[\cdot]$ denoting the indicator function:

| Metric (per LLM $m$) | Definition | Direction |
|---|---|---|
| Accuracy | $\frac1n\sum_i\mathbf1[\widehat z_{im}=z_{im}]$ | Higher |
| Recall for class $j$ | Fraction of true class-$j$ prompts predicted as class $j$ | Higher |
| Balanced accuracy | Mean recall over true classes present in the evaluated rows | Higher |
| NLL (negative log-likelihood) | $-\frac1n\sum_i\log\max(p_{im,z_{im}},10^{-12})$ | Lower |
| Zero true-probability rate | $\frac1n\sum_i\mathbf1[p_{im,z_{im}}=0]$, before the NLL floor | Lower |
| MSE | $\frac1n\sum_i(\widehat y_{im}-y_{im})^2$ | Lower |
| MAE | $\frac1n\sum_i\lvert\widehat y_{im}-y_{im}\rvert$ | Lower |
| Cost WAPE | $\frac{\sum_i\lvert\widehat c_{im}-c_{im}\rvert}{\sum_i c_{im}}$ | Lower |

Here $y$ denotes an original score or dollar cost. NLL uses the natural
logarithm. Accuracy measures score-class prediction. WAPE is used only
for costs; `0.10` means 10%. With all five classes present, balanced
accuracy equals one fifth of the sum of their recalls.

The next two code cells separate **TF-IDF** and **E5_388** CV preprocessing. Both use the grouped folds created after loading train. Shared metric functions are defined with the model helpers below.

In [4]:
# TF-IDF: fit a fresh vectorizer on the fitting rows of each training fold.
texts = train_questions["question_text"].to_numpy()

def iter_tfidf_cv_features():
    for fold_number, (fit_rows, heldout_rows) in enumerate(folds, 1):
        vectorizer = clone(tfidf_vectorizer)
        X_fit = vectorizer.fit_transform(texts[fit_rows]).astype(np.float32).tocsr()
        X_heldout = vectorizer.transform(texts[heldout_rows]).astype(np.float32).tocsr()
        yield fold_number, fit_rows, heldout_rows, X_fit, X_heldout

In [5]:
# E5_388: keep semantic embeddings fixed and refit the auxiliary scalers per fold.
e5 = X_train["e5_388"]
raw_lengths = length_scaler.inverse_transform(e5[:, 384:385])
raw_counts = count_scaler.inverse_transform(e5[:, 385:387])

def iter_e5_cv_features():
    for fold_number, (fit_rows, heldout_rows) in enumerate(folds, 1):
        lengths = clone(length_scaler).fit(raw_lengths[fit_rows])
        counts = clone(count_scaler).fit(raw_counts[fit_rows])
        def transform(rows):
            return np.column_stack([e5[rows, :384], lengths.transform(raw_lengths[rows]),
                                    counts.transform(raw_counts[rows]), e5[rows, 387:388]]).astype(np.float32)
        X_fit, X_heldout = transform(fit_rows), transform(heldout_rows)
        yield fold_number, fit_rows, heldout_rows, X_fit, X_heldout

def iter_cv_features(embedding):
    # Keep the same interface for the CV searches below.
    builders = {"tfidf": iter_tfidf_cv_features, "e5_388": iter_e5_cv_features}
    return builders[embedding]()

In [6]:
K_VALUES = [1, 2, 3, 5, 10, 20, 40, 60, 80, 100]

## kNN: class frequencies and mean dollar costs

Cosine distance retrieves similar fitting prompts:

$$d(x,x')=1-\frac{x^{\mathsf T}x'}{\|x\|\|x'\|}.$$

For LLM $m$, class $j$, and neighbor indices $N_k(x)$,

$$p_{mj}^{(k)}(x)=\frac1k\sum_{i\in N_k(x)}\mathbf1[z_{im}=j],
\qquad \widehat c_m^{(k)}(x)=\frac1k\sum_{i\in N_k(x)}c_{im}.$$

All LLMs share the retrieved prompt indices. Retrieve the largest
neighborhood once per batch and reuse its prefixes for every $k$.
Select score and cost neighbor counts independently.

In [7]:
# Shared evaluation helpers for CV and training diagnostics.
def score_metrics(truth, classes, probabilities):
    assert probabilities.shape == (*truth.shape, NUM_CLASSES)
    assert np.isfinite(probabilities).all() and (probabilities >= 0).all()
    assert np.allclose(probabilities.sum(axis=-1), 1, atol=1e-6)
    predicted = probabilities.argmax(axis=-1)
    true_p = np.take_along_axis(probabilities, classes[..., None], axis=-1)[..., 0]
    result = pd.DataFrame({
        "Accuracy": (predicted == classes).mean(axis=0),
        "Balanced_accuracy": [np.mean([(predicted[classes[:, m] == j, m] == j).mean()
                                      for j in np.unique(classes[:, m])]) for m in range(len(model_names))],
        "NLL": -np.log(np.maximum(true_p, 1e-12)).mean(axis=0),
        "Zero_true_probability_rate": (true_p == 0).mean(axis=0),
    }, index=pd.Index(model_names, name="Model"))
    for label, values in {"Hard": score_support[predicted], "Expected": probabilities @ score_support}.items():
        errors = values - truth
        result[f"{label}_MSE"] = np.square(errors).mean(axis=0)
        result[f"{label}_MAE"] = np.abs(errors).mean(axis=0)
    return result

def cost_metrics(truth, predictions):
    assert predictions.shape == truth.shape and np.isfinite(predictions).all() and (predictions >= 0).all()
    errors = predictions - truth
    return pd.DataFrame({"MSE": np.square(errors).mean(axis=0), "MAE": np.abs(errors).mean(axis=0),
                         "WAPE": np.abs(errors).sum(axis=0) / truth.sum(axis=0)},
                        index=pd.Index(model_names, name="Model"))

KNN_QUERY_BATCH_SIZE = 128


def knn_predict_arrays(index, query_features, reference_classes,
                       reference_costs, requested_k, query_name):
    n, m = query_features.shape[0], len(model_names)
    predictions = {
        k: {"probability": np.empty((n, m, NUM_CLASSES), dtype=np.float64),
            "cost": np.empty((n, m), dtype=np.float64)}
        for k in sorted(set(requested_k))
    }
    for batch, start in enumerate(range(0, n, KNN_QUERY_BATCH_SIZE), 1):
        stop = min(start + KNN_QUERY_BATCH_SIZE, n)
        # Max-grid retrieval preserves the original prefix/tie behavior.
        neighbors = index.kneighbors(query_features[start:stop],
                                     n_neighbors=max(K_VALUES), return_distance=False)
        labels = reference_classes[neighbors]
        cost_sums = np.cumsum(reference_costs[neighbors], axis=1, dtype=np.float64)
        counts = np.zeros((stop - start, m, NUM_CLASSES), dtype=np.float64)
        rows, models = np.indices((stop - start, m))
        for position in range(max(K_VALUES)):
            counts[rows, models, labels[:, position, :]] += 1
            k = position + 1
            if k in predictions:
                predictions[k]["probability"][start:stop] = counts / k
                predictions[k]["cost"][start:stop] = cost_sums[:, position, :] / k
        if batch % 10 == 0:
            print(f"kNN / {query_name}: {stop}/{n} prompts", flush=True)
    return predictions


def knn_oof_metrics(probabilities, predicted_costs, scores, classes, costs):
    return pd.concat([
        score_metrics(scores, classes, probabilities),
        cost_metrics(costs, predicted_costs).add_prefix("Cost_"),
    ], axis=1).rename_axis("Model")

def store_embedding_table(name, embedding, frame):
    """Replace this representation's rows while retaining the other one's results."""
    previous = cv_tables.get(name, pd.DataFrame(columns=frame.columns))
    other = previous.loc[previous["Embedding"].ne(embedding)]
    combined = pd.concat([other, frame], ignore_index=True) if len(other) else frame.copy()
    order = combined["Embedding"].map({"tfidf": 0, "e5_388": 1})
    cv_tables[name] = combined.iloc[np.argsort(order.to_numpy(), kind="stable")].reset_index(drop=True)

def invalidate_embedding_fit(embedding):
    """After CV, require a fresh fit for this representation before export."""
    global training_diagnostics_current
    training_diagnostics_current = False
    fitted_models.pop(embedding, None)
    prediction_bundles.pop(embedding, None)
    if "final_fit_diagnostics" in cv_tables:
        table = cv_tables["final_fit_diagnostics"]
        cv_tables["final_fit_diagnostics"] = table.loc[table["Embedding"].ne(embedding)].copy()

## Select two neighbor counts with training-only CV

Use three shuffled `GroupKFold` folds and fold-local preprocessing.
Each OOF prompt's whole group is excluded from its neighbor library.
Search $k\in\{1,2,3,5,10,20,40,60,80,100\}$.

Pool OOF training rows, compute per-LLM metrics, then average LLMs
equally. Maximize mapped-class Accuracy and minimize dollar-cost WAPE:

$$A(k)=\frac1M\sum_m\frac1N\sum_i
\mathbf1[\widehat z_{im}^{(k)}=z_{im}],\qquad
W(k)=\frac1M\sum_m
\frac{\sum_i|\widehat c_{im}^{(k)}-c_{im}|}{\sum_i c_{im}}.$$

Exact ties favor larger $k$. NLL and numerical score errors are
diagnostics, not selection criteria. Export numeric summaries and
per-LLM results; full OOF arrays are only used in memory.

In [8]:
# Shared CV implementation; execution is split by representation below.
def run_knn_cv(embedding):
    assert max(K_VALUES) <= min(len(rows) for rows, _ in folds), (
        "The largest k must not exceed the fitting-row count in any CV fold."
    )
    knn_scores = train_scores.loc[:, model_names].to_numpy(dtype=np.float64)
    knn_costs = train_costs.loc[:, model_names].to_numpy(dtype=np.float64)
    knn_classes = np.asarray(train_class_ids)
    knn_n, knn_m = knn_scores.shape
    knn_fold_rows, knn_summaries, knn_per_llm, knn_selections = [], [], [], []
    oof_probabilities = np.empty((len(K_VALUES), knn_n, knn_m, NUM_CLASSES))
    oof_costs = np.empty((len(K_VALUES), knn_n, knn_m))
    for fold, fit_idx, hold_idx, X_fit, X_hold in iter_cv_features(embedding):
        print(f"kNN training CV / {embedding} / fold {fold}: "
              f"{len(fit_idx)} fitting, {len(hold_idx)} OOF prompts", flush=True)
        started = perf_counter()
        index = NearestNeighbors(metric="cosine", algorithm="brute", n_jobs=-1).fit(X_fit)
        predictions = knn_predict_arrays(index, X_hold, knn_classes[fit_idx],
                                         knn_costs[fit_idx], K_VALUES,
                                         f"{embedding} CV fold {fold}")
        for position, k in enumerate(K_VALUES):
            oof_probabilities[position, hold_idx] = predictions[k]["probability"]
            oof_costs[position, hold_idx] = predictions[k]["cost"]
        knn_fold_rows.append({"Embedding": embedding, "Fold": fold,
                              "Fit_questions": len(fit_idx), "Heldout_questions": len(hold_idx),
                              "Seconds": perf_counter() - started})
        del index, predictions, X_fit, X_hold
    rows = []
    for position, k in enumerate(K_VALUES):
        # Pooled metrics preserve WAPE denominators and class recalls.
        metrics = knn_oof_metrics(oof_probabilities[position], oof_costs[position],
                                  knn_scores, knn_classes, knn_costs)
        rows.append({"Embedding": embedding, "k": k,
                     **metrics.mean().add_prefix("CV_").to_dict()})
        knn_per_llm.append(metrics.add_prefix("CV_").reset_index().assign(
            Embedding=embedding, k=k))
    summary = pd.DataFrame(rows)
    score_best = summary.sort_values(["CV_Accuracy", "k"],
                                     ascending=[False, False], kind="stable").iloc[0]
    cost_best = summary.sort_values(["CV_Cost_WAPE", "k"],
                                    ascending=[True, False], kind="stable").iloc[0]
    score_k, cost_k = int(score_best["k"]), int(cost_best["k"])
    selected_parameters[embedding] = {"score_k": score_k, "cost_k": cost_k}
    summary["Selected_score"] = summary["k"].eq(score_k)
    summary["Selected_cost"] = summary["k"].eq(cost_k)
    knn_summaries.append(summary)
    knn_selections.append({"Embedding": embedding, "Selected_score_k": score_k,
                           "Selected_cost_k": cost_k, "Selected_CV_Accuracy": score_best["CV_Accuracy"],
                           "Selected_CV_Cost_WAPE": cost_best["CV_Cost_WAPE"]})
    print(f"Selected kNN / {embedding}: score k={score_k}, cost k={cost_k}", flush=True)
    del oof_probabilities, oof_costs, metrics

    store_embedding_table("knn_cv_folds", embedding, pd.DataFrame(knn_fold_rows))
    store_embedding_table("knn_cv_summary", embedding, pd.concat(knn_summaries, ignore_index=True))
    store_embedding_table("knn_cv_per_llm", embedding, pd.concat(knn_per_llm, ignore_index=True))
    store_embedding_table("knn_cv_selection", embedding, pd.DataFrame(knn_selections))
    invalidate_embedding_fit(embedding)
    return summary

### kNN cross-validation — TF-IDF

In [9]:
# kNN cross-validation: TF-IDF only.
display(run_knn_cv("tfidf"))

kNN training CV / tfidf / fold 1: 14601 fitting, 7299 OOF prompts


kNN / tfidf CV fold 1: 1280/7299 prompts


kNN / tfidf CV fold 1: 2560/7299 prompts


kNN / tfidf CV fold 1: 3840/7299 prompts


kNN / tfidf CV fold 1: 5120/7299 prompts


kNN / tfidf CV fold 1: 6400/7299 prompts


... (10 outputs left out)

kNN / tfidf CV fold 3: 5120/7303 prompts


kNN / tfidf CV fold 3: 6400/7303 prompts


Selected kNN / tfidf: score k=80, cost k=5


,Embedding,k,CV_Accuracy,CV_Balanced_accuracy,CV_NLL,CV_Zero_true_probability_rate,CV_Hard_MSE,CV_Hard_MAE,CV_Expected_MSE,CV_Expected_MAE,CV_Cost_MSE,CV_Cost_MAE,CV_Cost_WAPE,Selected_score,Selected_cost
0,tfidf,1,0.603919,0.433863,10.944132,0.396081,0.299439,0.324698,0.299439,0.324698,7.012447e-07,0.000231,0.256322,False,False
1,tfidf,2,0.584093,0.431876,6.273739,0.218012,0.311810,0.338566,0.227269,0.325022,6.165273e-07,0.000219,0.244504,False,False
2,tfidf,3,0.632428,0.435402,4.224835,0.139000,0.282043,0.305383,0.202460,0.325034,5.879118e-07,0.000215,0.240323,False,False
3,tfidf,5,0.646218,0.436909,2.545455,0.073769,0.269886,0.292620,0.182720,0.325883,5.605857e-07,0.000213,0.239518,False,True
4,tfidf,10,0.660506,0.439225,1.370337,0.027862,0.260587,0.282286,0.167940,0.326702,5.456934e-07,0.000215,0.242519,False,False
5,tfidf,20,0.670033,0.437532,0.916521,0.010025,0.251260,0.273029,0.161287,0.328437,5.481534e-07,0.000222,0.252175,False,False
6,tfidf,40,0.671457,0.434394,0.759978,0.003861,0.248429,0.270695,0.158622,0.330337,5.530765e-07,0.000232,0.267108,False,False
7,tfidf,60,0.672088,0.433412,0.720741,0.002329,0.247289,0.269736,0.157960,0.331490,5.541103e-07,0.000238,0.275590,False,False
8,tfidf,80,0.672154,0.432045,0.706106,0.001793,0.246928,0.269470,0.157785,0.332301,5.558821e-07,0.000242,0.281379,True,False
9,tfidf,100,0.671988,0.430709,0.699881,0.001565,0.247526,0.269929,0.157770,0.333171,5.575412e-07,0.000245,0.285860,False,False


### kNN cross-validation — E5_388

In [10]:
# kNN cross-validation: E5_388 only.
display(run_knn_cv("e5_388"))

kNN training CV / e5_388 / fold 1: 14601 fitting, 7299 OOF prompts


kNN / e5_388 CV fold 1: 1280/7299 prompts


kNN / e5_388 CV fold 1: 2560/7299 prompts


kNN / e5_388 CV fold 1: 3840/7299 prompts


kNN / e5_388 CV fold 1: 5120/7299 prompts


kNN / e5_388 CV fold 1: 6400/7299 prompts


... (10 outputs left out)

kNN / e5_388 CV fold 3: 5120/7303 prompts


kNN / e5_388 CV fold 3: 6400/7303 prompts


Selected kNN / e5_388: score k=80, cost k=10


,Embedding,k,CV_Accuracy,CV_Balanced_accuracy,CV_NLL,CV_Zero_true_probability_rate,CV_Hard_MSE,CV_Hard_MAE,CV_Expected_MSE,CV_Expected_MAE,CV_Cost_MSE,CV_Cost_MAE,CV_Cost_WAPE,Selected_score,Selected_cost
0,e5_388,1,0.606326,0.440901,10.877607,0.393674,0.298880,0.323551,0.298880,0.323551,3.296510e-07,0.000126,0.135042,False,False
1,e5_388,2,0.584579,0.438510,6.076425,0.210623,0.312365,0.339194,0.224499,0.322763,2.518098e-07,0.000110,0.118154,False,False
2,e5_388,3,0.632694,0.445693,3.950328,0.128580,0.279869,0.303455,0.199681,0.323074,2.276482e-07,0.000104,0.111601,False,False
3,e5_388,5,0.646588,0.451477,2.235343,0.061984,0.268524,0.291074,0.180322,0.323709,2.183501e-07,0.000100,0.106965,False,False
4,e5_388,10,0.660419,0.452911,1.136493,0.019228,0.260789,0.281998,0.166039,0.325250,2.130314e-07,0.000097,0.103515,False,True
5,e5_388,20,0.672237,0.450059,0.794717,0.006032,0.252311,0.272823,0.159693,0.327796,2.310515e-07,0.000099,0.103994,False,False
6,e5_388,40,0.677891,0.448435,0.703484,0.002574,0.247531,0.267843,0.157162,0.330444,2.523957e-07,0.000101,0.105465,False,False
7,e5_388,60,0.678912,0.446029,0.686044,0.001860,0.246529,0.266871,0.156753,0.332125,2.669221e-07,0.000103,0.106804,False,False
8,e5_388,80,0.679195,0.444098,0.679317,0.001532,0.246229,0.266602,0.156852,0.333454,2.820978e-07,0.000105,0.108557,True,False
9,e5_388,100,0.678746,0.441739,0.674374,0.001262,0.246383,0.266879,0.156962,0.334294,2.979142e-07,0.000107,0.110179,False,False


## Fit the selected predictors and predict training rows

Build one full-training cosine index per representation. Combine
probabilities from $k_{\rm score}$ with dollar costs from $k_{\rm cost}$.

**Final metrics are in-sample training errors.** Each query remains in
its own training-neighbor library; self is not excluded. Distance ties
can affect which equal-distance prompts are retrieved. These optimistic
errors do not estimate generalization. The CV tables above use only
out-of-fold training predictions.

In [11]:
# Shared final-fit implementation; defining this function does not train a model.
def fit_knn_embedding(embedding):
    invalidate_embedding_fit(embedding)
    knn_classes = train_class_ids
    knn_costs = train_cost_values
    score_k = selected_parameters[embedding]["score_k"]
    cost_k = selected_parameters[embedding]["cost_k"]
    index = NearestNeighbors(metric="cosine", algorithm="brute", n_jobs=-1).fit(X_train[embedding])
    predictions = knn_predict_arrays(index, X_train[embedding], knn_classes, knn_costs,
                                     [score_k, cost_k], f"{embedding} in-sample training")
    prediction_bundles[embedding] = {"probability": predictions[score_k]["probability"],
                                    "cost": predictions[cost_k]["cost"]}
    fitted_models[embedding] = {"neighbor_index": index, "score_k": score_k, "cost_k": cost_k,
                                "reference_class_ids": knn_classes.copy(),
                                "reference_costs_usd": knn_costs.copy(),
                                "query_max_k": max(K_VALUES), "self_neighbor_excluded": False}
    del predictions
    print(f"Final kNN fit and training predictions complete: {embedding}", flush=True)

### kNN final training — TF-IDF

In [12]:
# kNN final training: TF-IDF only.
fit_knn_embedding("tfidf")

kNN / tfidf in-sample training: 1280/21900 prompts


kNN / tfidf in-sample training: 2560/21900 prompts


kNN / tfidf in-sample training: 3840/21900 prompts


kNN / tfidf in-sample training: 5120/21900 prompts


kNN / tfidf in-sample training: 6400/21900 prompts


kNN / tfidf in-sample training: 7680/21900 prompts


... (8 outputs left out)

kNN / tfidf in-sample training: 19200/21900 prompts


kNN / tfidf in-sample training: 20480/21900 prompts


kNN / tfidf in-sample training: 21760/21900 prompts


Final kNN fit and training predictions complete: tfidf


### kNN final training — E5_388

In [13]:
# kNN final training: E5_388 only.
fit_knn_embedding("e5_388")

kNN / e5_388 in-sample training: 1280/21900 prompts


kNN / e5_388 in-sample training: 2560/21900 prompts


kNN / e5_388 in-sample training: 3840/21900 prompts


kNN / e5_388 in-sample training: 5120/21900 prompts


kNN / e5_388 in-sample training: 6400/21900 prompts


kNN / e5_388 in-sample training: 7680/21900 prompts


... (8 outputs left out)

kNN / e5_388 in-sample training: 19200/21900 prompts


kNN / e5_388 in-sample training: 20480/21900 prompts


kNN / e5_388 in-sample training: 21760/21900 prompts


Final kNN fit and training predictions complete: e5_388


## Training diagnostics and baseline

The training-frequency baseline ignores the prompt. For LLM $m$, use
class frequencies $q_{mj}=N^{-1}\sum_i\mathbf1[z_{im}=j]$ and mean cost
$\overline c_m=N^{-1}\sum_i c_{im}$ for every row. Hard and expected
baseline scores follow the same formulas as the fitted models.

Report both per-LLM results and an equal-LLM summary. Training-fit errors
reveal fit quality, including overfitting; they do not measure routing
savings or held-out performance. CV already chose hyperparameters, so
its selected results are also not an independent final test.

In [14]:
training_diagnostics_current = False
assert set(prediction_bundles) == set(X_train), "Run both final-training cells before the shared diagnostics."
frequencies = np.stack([(train_class_ids == j).mean(axis=0) for j in range(NUM_CLASSES)], axis=-1)
baseline = {
    "probability": np.broadcast_to(frequencies, (*train_class_ids.shape, NUM_CLASSES)),
    "cost": np.broadcast_to(train_cost_values.mean(axis=0), train_cost_values.shape),
}
score_tables, cost_tables = [], []
for embedding, bundle in {"Training frequency": baseline, **prediction_bundles}.items():
    score_tables.append(score_metrics(train_score_values, train_class_ids, bundle["probability"])
                        .reset_index().assign(Embedding=embedding))
    cost_tables.append(cost_metrics(train_cost_values, bundle["cost"])
                       .add_prefix("Cost_").reset_index().assign(Embedding=embedding))
train_score_metrics = pd.concat(score_tables, ignore_index=True)
train_cost_metrics = pd.concat(cost_tables, ignore_index=True)
train_score_summary = train_score_metrics.groupby("Embedding", sort=False).mean(numeric_only=True)
train_cost_summary = train_cost_metrics.groupby("Embedding", sort=False).mean(numeric_only=True)
print("In-sample training metrics (equal-LLM averages)")
display(train_score_summary)
display(train_cost_summary)
training_diagnostics_current = True

In-sample training metrics (equal-LLM averages)


,Accuracy,Balanced_accuracy,NLL,Zero_true_probability_rate,Hard_MSE,Hard_MAE,Expected_MSE,Expected_MAE
Embedding,,,,,,,,
Training frequency,0.530992,0.200000,1.187953,0.0,0.305213,0.347055,0.178984,0.382029
tfidf,0.680623,0.439381,0.644806,0.0,0.239283,0.261564,0.153177,0.326749
e5_388,0.690008,0.454984,0.622906,0.0,0.236968,0.256814,0.151908,0.327292


,Cost_MSE,Cost_MAE,Cost_WAPE
Embedding,,,
Training frequency,1.825855e-06,0.000588,0.617193
tfidf,3.618946e-07,0.000167,0.185814
e5_388,1.707649e-07,0.000086,0.092460


## Save this run

Each notebook owns a separate folder under
`2_Router_Training/experiment_outputs/<notebook_name>/`.
`smoke/` and `full/` are separate; rerunning a mode overwrites its filenames.
Existing combined-notebook results stay in their original directory.

| Files | Contents |
|---|---|
| `*_cv_*.csv` | Candidate and per-LLM CV metrics; selected candidates are marked |
| `train_rows.csv` | Sample IDs, prompt groups, and CV fold, in prediction order |
| `train_score_*.csv`, `train_cost_*.csv` | Training metrics and summaries, including baseline |
| `<embedding>_train_class_probability.npy` | Shape: training prompts × 11 LLMs × 5 classes |
| `<embedding>_train_hard_score.pkl`, `..._expected_score.pkl`, `..._dollar_cost.pkl` | Labeled prediction tables |
| `<embedding>_model.joblib` | Complete fitted model package, including selected parameters, transforms, and column order |
| `feature_preprocessing.joblib` | Shared copy of the feature transforms |
| `score_anchor_coverage.csv`, `run_manifest.json` | Original training-score coverage and run metadata |

Prediction rows and LLM columns are saved in their original order.
Model artifacts enable a later, separate inference step. That step must
reproduce the same feature preparation; E5 encoding itself remains in
Step 1. This notebook exports training predictions only. Retain the
manifest when sharing results so smoke output is not confused with full.

In [15]:
assert set(fitted_models) == set(prediction_bundles) == set(X_train), "Run both final-training cells before saving."
assert training_diagnostics_current, "Rerun the shared training diagnostics after the latest CV or final fit before saving."
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
manifest = {
    "workflow": WORKFLOW, "mode": RUN_MODE, "status": "writing", "seed": SEED,
    "started_utc": run_started, "training_questions": len(train_questions),
    "data_splits_loaded": ["train"], "model_artifact_format": 1, "inference_deferred_to_next_step": True, "prediction_split": "train", "evaluation": "in_sample",
    "cv": "3-fold shuffled GroupKFold on question_group; preprocessing refitted per fold",
    "selected_parameters": selected_parameters, "score_support": score_support.tolist(),
    "model_names": model_names, "versions": {"sklearn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__},
}
manifest_path = OUTPUT_DIR / "run_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
for name, frame in cv_tables.items():
    frame.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)
train_questions[["sample_id", "question_group"]].assign(CV_fold=fold_ids).to_csv(OUTPUT_DIR / "train_rows.csv")
coverage.to_csv(OUTPUT_DIR / "score_anchor_coverage.csv", index=False)
for name, frame in {"train_score_per_llm": train_score_metrics, "train_cost_per_llm": train_cost_metrics}.items():
    frame.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)
train_score_summary.to_csv(OUTPUT_DIR / "train_score_summary.csv")
train_cost_summary.to_csv(OUTPUT_DIR / "train_cost_summary.csv")
for embedding, bundle in prediction_bundles.items():
    probabilities = bundle["probability"]
    np.save(OUTPUT_DIR / f"{embedding}_train_class_probability.npy", probabilities, allow_pickle=False)
    outputs = {"hard_score": score_support[probabilities.argmax(axis=-1)],
               "expected_score": probabilities @ score_support, "dollar_cost": bundle["cost"]}
    for name, values in outputs.items():
        pd.DataFrame(values, index=train_questions.index, columns=model_names).to_pickle(
            OUTPUT_DIR / f"{embedding}_train_{name}.pkl")
    feature_preprocessing = ({"tfidf_vectorizer": tfidf_vectorizer} if embedding == "tfidf" else
                             {"length_scaler": length_scaler, "count_scaler": count_scaler,
                              "semantic_dimension": 384,
                              "auxiliary_feature_order": ["length", "choice_count", "numeric_count", "code_indicator"]})
    artifact = {"format_version": 1, "method": 'kNN', "embedding": embedding,
                "run_mode": RUN_MODE, "training_questions": len(train_questions),
                "training_split": "train", "model": fitted_models[embedding],
                "model_names": model_names, "score_support": score_support,
                "selected_parameters": selected_parameters[embedding],
                "n_features": X_train[embedding].shape[1], "preprocessing": feature_preprocessing,
                "versions": manifest["versions"], "normalized_cost_floor": 1e-6}
    joblib.dump(artifact, OUTPUT_DIR / f"{embedding}_model.joblib")
joblib.dump({"tfidf_vectorizer": tfidf_vectorizer, "e5_length_scaler": length_scaler,
             "e5_count_scaler": count_scaler, "e5_dimension": 388}, OUTPUT_DIR / "feature_preprocessing.joblib")
manifest.update(status="complete", completed_utc=datetime.now(timezone.utc).isoformat(),
                files=sorted(path.name for path in OUTPUT_DIR.iterdir() if path.is_file()))
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Saved", len(manifest["files"]), "files to", OUTPUT_DIR)

Saved 22 files to C:\Users\johnh\MachineLearning\project_erdos\2_Router_Training\experiment_outputs\router_training_score_and_cost_kNN\full


## Reuse the saved model in the next step

kNN has no learned coefficient matrix. Its reusable model consists of the fitted training-neighbor index, aligned training class labels and dollar costs, and the selected score/cost neighbor counts. The package includes all of them.

Each representation has one complete package: `tfidf_model.joblib` or
`e5_388_model.joblib`. It includes the score anchors, LLM column order,
feature transforms, and smoke/full provenance. Use the `full/` artifacts
for the next project stage. Load them in an environment with compatible
package versions; neither CV nor fitting needs to be repeated.

The following cell **defines an inference helper only**. No new dataset
is loaded and no calibration or validation predictions are made here.
Copy the helper into the next notebook with `import numpy as np`.

In [16]:
def predict_saved_knn(artifact, features, batch_size=128):
    """Next-step inference on already prepared features; this cell makes no predictions."""
    if features.ndim != 2 or features.shape[1] != artifact["n_features"]:
        raise ValueError("Feature width must match the saved representation.")
    if batch_size < 1:
        raise ValueError("batch_size must be positive.")
    support = np.asarray(artifact["score_support"])
    probabilities = np.zeros((features.shape[0], len(artifact["model_names"]), len(support)))
    costs = np.empty(probabilities.shape[:2])
    if features.shape[0]:
        model = artifact["model"]
        score_k, cost_k = model["score_k"], model["cost_k"]
        for start in range(0, len(probabilities), batch_size):
            stop = min(start + batch_size, len(probabilities))
            neighbors = model["neighbor_index"].kneighbors(
                features[start:stop], n_neighbors=model["query_max_k"], return_distance=False)
            labels = model["reference_class_ids"][neighbors[:, :score_k]]
            probabilities[start:stop] = np.stack([(labels == j).mean(axis=1) for j in range(len(support))], axis=-1)
            costs[start:stop] = np.cumsum(model["reference_costs_usd"][neighbors], axis=1)[:, cost_k - 1] / cost_k
    return {"probability": probabilities, "hard_score": support[probabilities.argmax(axis=-1)],
            "expected_score": probabilities @ support, "dollar_cost": costs}